<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/Train_SNAP_FF_Ni.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install LAMMPS

In [ ]:
pip install scikit-learn==0.22.1

In [ ]:
import os
# %env LD_PRELOAD=
# !apt install cmake build-essential git ccache
# Clone the LAMMPS git
if not os.path.exists('lammps'):
   os.system('git clone https://github.com/lammps/lammps.git')


In [ ]:
os.chdir('lammps/src')
!make yes-MANYBODY
!make serial

In [ ]:
!export LMP_EXE=/content/lammps/src/lmp_serial

In [ ]:
import os
os.chdir('/content')

In [ ]:
!ls

In [ ]:
import mlearn

In [ ]:
os.makedirs('Software')
os.chdir('Software')
if not os.path.exists('mlearn'):
  !git clone https://github.com/materialsvirtuallab/mlearn.git
os.chdir('mlearn')
!pip install -e .
os.chdir('/content')

In [ ]:
!ls Software/mlearn/data/Ni

In [ ]:
from pymatgen.core import Structure
from pymatgen.core.lattice import Lattice
from pymatgen.core.composition import Element
from monty.serialization import loadfn

# data = loadfn('data.json')
data = loadfn("Software/mlearn/data/Ni/training.json")
train_structures = [d['structure'] for d in data]
train_energies = [d['outputs']['energy'] for d in data]
train_forces = [d['outputs']['forces'] for d in data]
train_stresses = [d['outputs']['virial_stress'] for d in data]


data = loadfn("Software/mlearn/data/Ni/test.json")
test_structures = [d['structure'] for d in data]
test_energies = [d['outputs']['energy'] for d in data]
test_forces = [d['outputs']['forces'] for d in data]
test_stresses = [d['outputs']['virial_stress'] for d in data]

Modify line 9 to from pymatgen.core.structure import Structure in data.py if you encounter error import Structure from pymatgen

In [ ]:
import numpy as np
from mlearn.data import pool_from, convert_docs

train_pool = pool_from(train_structures, train_energies, train_forces)
_, df = convert_docs(train_pool)

weights = np.ones(len(df['dtype']), )

# set the weights for energy equal to 100
weights[df['dtype'] == 'energy'] = 100


In [ ]:
# pip install scikit-learn==0.22.2.post1

# snap.py line 10:from pymatgen.core.periodic_table import Element


calcs.py line 21 from pymatgen.core.structure import Structure, Lattice, Element


In [ ]:
!/content/lammps/src/lmp_serial

In [ ]:
!cp /content/lammps/src/lmp_serial .

In [ ]:
!export LMP_EXE=lmp_serial

In [ ]:
import os

from mlearn.describers import BispectrumCoefficients
from mlearn.models import LinearModel
from mlearn.potentials.snap import SNAPotential
#from: https://github.com/materialsvirtuallab/maml/blob/master/mvl_models/pes/Si/snap.2020.01/SNAPotential.snapparam
element_profile = {'Ni': {'r': 0.5, 'w': 1.9}}
describer = BispectrumCoefficients(rcutfac=4.1, twojmax=8, element_profile=element_profile, 
                                   quadratic=False, pot_fit=True)
model = LinearModel(describer=describer)
snap = SNAPotential(model=model)
snap.train(train_structures, train_energies, train_forces, weights=weights,)

In [ ]:
import pymatgen

In [ ]:
from pymatgen.core.structure import Structure

In [ ]:
!ls -altr lammps/src